In [1]:
import os

os.makedirs("corpus", exist_ok=True)

documentos = {
    "empresa_alpha_2024.txt": """
        INFORME ANUAL EMPRESA ALPHA 2024

        Resultados financieros:
        El revenue total del ejercicio 2024 fue de 47.3 millones de euros, 
        representando un crecimiento del 18.4% respecto a los 39.9 millones de 2023.
        El EBITDA alcanzó 12.1 millones de euros, con margen del 25.6%.
        El beneficio neto fue de 6.8 millones de euros.
        La deuda neta se redujo a 3.2 millones de euros desde 8.1 millones en 2023.

        Segmentos de negocio:
        El segmento de software representó el 68% del revenue (32.2M euros).
        El segmento de servicios profesionales aportó el 32% restante (15.1M euros).
        El segmento de software creció un 24% interanual.

        Mercados geográficos:
        España representa el 45% de las ventas (21.3M euros).
        Europa central aportó el 35% (16.6M euros).
        Latinoamérica contribuyó el 20% restante (9.5M euros).

        Perspectivas 2025:
        La compañía espera un crecimiento del revenue entre el 15% y el 20%.
        Se prevé una inversión en I+D de 4.2 millones de euros.
        El objetivo de EBITDA margin es superar el 27%.
        """,

    "empresa_beta_2024.txt": """
        INFORME ANUAL EMPRESA BETA 2024

        Resultados financieros:
        Beta registró un revenue de 124.7 millones de euros en 2024.
        El crecimiento respecto a 2023 fue del 6.2% (117.4 millones en 2023).
        El EBITDA fue de 18.3 millones, con margen del 14.7%.
        El beneficio neto cayó a 2.1 millones desde 5.4 millones en 2023.
        La deuda neta aumentó a 31.4 millones de euros.

        Segmentos de negocio:
        Manufactura industrial: 78.6 millones de euros (63% del total).
        Distribución: 46.1 millones de euros (37% del total).
        La división de manufactura sufrió presión de márgenes por costes energéticos.

        Mercados geográficos:
        Mercado doméstico (España y Portugal): 89% de ventas.
        Exportación: 11% de ventas, principalmente Francia e Italia.

        Perspectivas 2025:
        La compañía anticipa un entorno desafiante por inflación de costes.
        El plan de reestructuración prevé reducción de plantilla del 8%.
        Objetivo de revenue: mantenerse entre 120 y 128 millones de euros.
        """,

    "empresa_gamma_2024.txt": """
        INFORME ANUAL EMPRESA GAMMA 2024

        Resultados financieros:
        Gamma alcanzó un revenue de 8.9 millones de euros, crecimiento del 67% vs 2023 (5.3M).
        La compañía sigue en fase de inversión: EBITDA negativo de -2.4 millones.
        El beneficio neto fue de -3.1 millones de euros.
        La caja disponible es de 14.2 millones de euros tras ronda de financiación Serie B.

        Modelo de negocio:
        Gamma opera como plataforma SaaS de gestión de flota para empresas de transporte.
        ARR (Annual Recurring Revenue) de 7.8 millones, crecimiento del 71% interanual.
        NRR (Net Revenue Retention) del 118%, indicando expansión en clientes existentes.
        Churn rate del 4.2% anual.

        Métricas operativas:
        Clientes activos: 342 empresas.
        Empleados: 89 personas, +31 vs año anterior.
        Coste de adquisición de cliente (CAC): 18.400 euros.
        Lifetime Value (LTV): 94.000 euros. Ratio LTV/CAC: 5.1x.

        Perspectivas 2025:
        Objetivo ARR: 13 millones de euros a cierre de 2025.
        Expansión a mercado francés y alemán en Q2 2025.
        Previsión de alcanzar EBITDA breakeven en Q4 2025.
        """
}

for filename, content in documentos.items():
    with open(f"corpus/{filename}", "w", encoding="utf-8") as f:
        f.write(content)

print("Corpus creado:")
for f in os.listdir("corpus"):
    print(f"  {f}")

Corpus creado:
  empresa_alpha_2024.txt
  empresa_beta_2024.txt
  empresa_gamma_2024.txt


In [ ]:
import chromadb
from chromadb.utils import embedding_functions
import anthropic
from dotenv import load_dotenv

load_dotenv()
client_anthropic = anthropic.Anthropic()

# Chroma con embeddings de sentence-transformers (gratis, sin API key)
chroma_client = chromadb.Client()
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

collection = chroma_client.create_collection(
    name="informes",
    embedding_function=ef
)

# Chunking manual: dividir cada documento en fragmentos
def chunk_text(text, chunk_size=200, overlap=50):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

# Indexar todos los documentos
all_chunks = []
all_ids = []
all_metadata = []

for filename in os.listdir("corpus"):
    with open(f"corpus/{filename}", encoding="utf-8") as f:
        text = f.read()
    chunks = chunk_text(text)
    for j, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        all_ids.append(f"{filename}_chunk_{j}")
        all_metadata.append({"source": filename, "chunk_index": j})

collection.add(
    documents=all_chunks,
    ids=all_ids,
    metadatas=all_metadata
)

print(f"Indexados {len(all_chunks)} chunks de {len(os.listdir('corpus'))} documentos")

In [ ]:
def rag_query(pregunta, n_results=3, verbose=True):
    # 1. Retrieval
    results = collection.query(query_texts=[pregunta], n_results=n_results)
    chunks_recuperados = results["documents"][0]
    fuentes = [m["source"] for m in results["metadatas"][0]]
    
    if verbose:
        print(f"Pregunta: {pregunta}")
        print(f"\nChunks recuperados:")
        for i, (chunk, fuente) in enumerate(zip(chunks_recuperados, fuentes)):
            print(f"  [{i+1}] {fuente}: '{chunk[:100]}...'")
    
    # 2. Generation
    contexto = "\n\n---\n\n".join(chunks_recuperados)
    prompt = f"""
        Basándote únicamente en los siguientes fragmentos de informes anuales, responde la pregunta.

        Si la información no está en los fragmentos, dilo explícitamente.

        ======================
        FRAGMENTOS
        ======================
        {contexto}

        ======================
        PREGUNTA
        ======================
        {pregunta}

        ======================
        RESPUESTA
        ======================
    """
    
    response = client_anthropic.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=400,
        messages=[{"role": "user", "content": prompt}]
    )
    
    respuesta = response.content[0].text
    if verbose:
        print(f"\nRespuesta del modelo:\n{respuesta}")
    return respuesta

# Caso donde RAG funciona bien
rag_query("¿Cuál fue el revenue de empresa Alpha en 2024?")

Pregunta: ¿Cuál fue el revenue de empresa Alpha en 2024?

Chunks recuperados:
  [1] empresa_alpha_2024.txt: 'INFORME ANUAL EMPRESA ALPHA 2024 Resultados financieros: El revenue total del ejercicio 2024 fue de ...'
  [2] empresa_gamma_2024.txt: 'INFORME ANUAL EMPRESA GAMMA 2024 Resultados financieros: Gamma alcanzó un revenue de 8.9 millones de...'
  [3] empresa_beta_2024.txt: 'INFORME ANUAL EMPRESA BETA 2024 Resultados financieros: Beta registró un revenue de 124.7 millones d...'


/tmp/ipykernel_33416/2274580506.py:35: DeprecationWarning: The model 'claude-sonnet-4-20250514' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client_anthropic.messages.create(



Respuesta del modelo:
Según el fragmento del informe anual de la Empresa Alpha 2024, el revenue total del ejercicio 2024 fue de **47.3 millones de euros**, lo que representó un crecimiento del 18.4% respecto a los 39.9 millones de euros registrados en 2023.


'Según el fragmento del informe anual de la Empresa Alpha 2024, el revenue total del ejercicio 2024 fue de **47.3 millones de euros**, lo que representó un crecimiento del 18.4% respecto a los 39.9 millones de euros registrados en 2023.'

In [ ]:
print("=" * 60)
print("FALLO 1: Pregunta que cruza información de dos chunks")
print("=" * 60)
rag_query("¿Qué empresa tiene mejor ratio LTV/CAC y cuál es su margen EBITDA?", n_results=1)

# Por qué falla: LTV/CAC está en un chunk de Gamma, margen EBITDA en otro.
# El retrieval recupera chunks de empresas distintas y el modelo mezcla datos.
# Cambia n_results=3 y observa si mejora.

FALLO 1: Pregunta que cruza información de dos chunks
Pregunta: ¿Qué empresa tiene mejor ratio LTV/CAC y cuál es su margen EBITDA?

Chunks recuperados:
  [1] empresa_gamma_2024.txt: 'INFORME ANUAL EMPRESA GAMMA 2024 Resultados financieros: Gamma alcanzó un revenue de 8.9 millones de...'


/tmp/ipykernel_33416/2274580506.py:35: DeprecationWarning: The model 'claude-sonnet-4-20250514' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client_anthropic.messages.create(



Respuesta del modelo:
Basándome únicamente en los fragmentos proporcionados, no puedo responder completamente a tu pregunta.

Los fragmentos solo contienen información de **una empresa (Gamma)**, que tiene:
- Ratio LTV/CAC: 5.1x
- EBITDA: -2.4 millones de euros (negativo)

Para determinar qué empresa tiene el mejor ratio LTV/CAC, necesitaría información de al menos dos empresas para poder hacer la comparación, pero solo se proporciona información de Empresa Gamma.

La información sobre otras empresas no está disponible en los fragmentos proporcionados.


'Basándome únicamente en los fragmentos proporcionados, no puedo responder completamente a tu pregunta.\n\nLos fragmentos solo contienen información de **una empresa (Gamma)**, que tiene:\n- Ratio LTV/CAC: 5.1x\n- EBITDA: -2.4 millones de euros (negativo)\n\nPara determinar qué empresa tiene el mejor ratio LTV/CAC, necesitaría información de al menos dos empresas para poder hacer la comparación, pero solo se proporciona información de Empresa Gamma.\n\nLa información sobre otras empresas no está disponible en los fragmentos proporcionados.'

In [8]:
print("=" * 60)
print("FALLO 2: Pregunta comparativa entre documentos")
print("=" * 60)
rag_query("¿Qué empresa creció más en revenue durante 2024?", n_results=1)

# Por qué falla: para comparar necesita ver los tres documentos a la vez.
# El retrieval solo recupera 1 chunk, probablemente del mismo documento.
# Cambia n_results=3 y observa si mejora.

FALLO 2: Pregunta comparativa entre documentos
Pregunta: ¿Qué empresa creció más en revenue durante 2024?

Chunks recuperados:
  [1] empresa_alpha_2024.txt: 'INFORME ANUAL EMPRESA ALPHA 2024 Resultados financieros: El revenue total del ejercicio 2024 fue de ...'


/tmp/ipykernel_33416/2274580506.py:35: DeprecationWarning: The model 'claude-sonnet-4-20250514' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client_anthropic.messages.create(



Respuesta del modelo:
La información no está en los fragmentos proporcionados.

Los fragmentos únicamente contienen información sobre la empresa Alpha y su crecimiento de revenue del 18.4% durante 2024. Para poder responder qué empresa creció más en revenue durante 2024, necesitaría información sobre otras empresas y sus respectivos crecimientos para poder hacer una comparación.


'La información no está en los fragmentos proporcionados.\n\nLos fragmentos únicamente contienen información sobre la empresa Alpha y su crecimiento de revenue del 18.4% durante 2024. Para poder responder qué empresa creció más en revenue durante 2024, necesitaría información sobre otras empresas y sus respectivos crecimientos para poder hacer una comparación.'

In [13]:
print("=" * 60)
print("FALLO 3: Chunk size demasiado pequeño rompe el contexto")
print("=" * 60)

# Crea una colección con chunks muy pequeños

chroma_client.delete_collection("informes_small_chunks")
collection_small = chroma_client.create_collection(
    name="informes_small_chunks",
    embedding_function=ef
)

all_chunks_small = []
all_ids_small = []

for filename in os.listdir("corpus"):
    with open(f"corpus/{filename}", encoding="utf-8") as f:
        text = f.read()
    # Chunks de solo 10 palabras, sin overlap
    chunks = chunk_text(text, chunk_size=10, overlap=0)
    for j, chunk in enumerate(chunks):
        all_chunks_small.append(chunk)
        all_ids_small.append(f"{filename}_small_{j}")

collection_small.add(documents=all_chunks_small, ids=all_ids_small)

# La misma pregunta con chunks pequeños
results = collection_small.query(
    query_texts=["¿Cuál fue el crecimiento de revenue de Alpha?"],
    n_results=3
)
print("Chunks recuperados con chunk_size=10:")
for chunk in results["documents"][0]:
    print(f"  '{chunk}'")
    
# Observa: los chunks son tan pequeños que pierden contexto.
# "18.4%" aparece sin la frase que explica qué es ese porcentaje.

FALLO 3: Chunk size demasiado pequeño rompe el contexto
Chunks recuperados con chunk_size=10:
  'millones, crecimiento del 71% interanual. NRR (Net Revenue Retention) del'
  'un crecimiento del 18.4% respecto a los 39.9 millones de'
  'INFORME ANUAL EMPRESA ALPHA 2024 Resultados financieros: El revenue total'


In [14]:
print("=" * 60)
print("FALLO 4: Alucinación numérica")
print("=" * 60)

# Pregunta con un número real en los documentos
respuesta = rag_query(
    "¿Cuál es exactamente el beneficio neto de Gamma en 2024?",
    verbose=False
)
print(f"Respuesta RAG: {respuesta}")
print("\nValor real en el documento: -3.1 millones de euros")
print("Observa si el modelo reproduce el número exacto o lo redondea/altera")

# Ahora fuerza una alucinación con información ausente
print("\n--- Pregunta sobre dato que NO existe ---")
rag_query("¿Cuál fue el revenue de empresa Alpha en el Q3 de 2024?")
# El modelo debería decir que no tiene esa información.
# Si no lo dice, está alucinando.

FALLO 4: Alucinación numérica


/tmp/ipykernel_33416/2274580506.py:35: DeprecationWarning: The model 'claude-sonnet-4-20250514' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client_anthropic.messages.create(


Respuesta RAG: Basándome en los fragmentos proporcionados, el beneficio neto de Gamma en 2024 fue de **-3.1 millones de euros**.

Esta información aparece explícitamente en el informe anual de la empresa Gamma 2024, en la sección de resultados financieros.

Valor real en el documento: -3.1 millones de euros
Observa si el modelo reproduce el número exacto o lo redondea/altera

--- Pregunta sobre dato que NO existe ---
Pregunta: ¿Cuál fue el revenue de empresa Alpha en el Q3 de 2024?

Chunks recuperados:
  [1] empresa_alpha_2024.txt: 'INFORME ANUAL EMPRESA ALPHA 2024 Resultados financieros: El revenue total del ejercicio 2024 fue de ...'
  [2] empresa_gamma_2024.txt: 'INFORME ANUAL EMPRESA GAMMA 2024 Resultados financieros: Gamma alcanzó un revenue de 8.9 millones de...'
  [3] empresa_beta_2024.txt: 'INFORME ANUAL EMPRESA BETA 2024 Resultados financieros: Beta registró un revenue de 124.7 millones d...'

Respuesta del modelo:
Basándome únicamente en los fragmentos proporcionados, **la in

'Basándome únicamente en los fragmentos proporcionados, **la información sobre el revenue de la empresa Alpha en el Q3 de 2024 no está disponible**.\n\nLos fragmentos solo contienen información sobre el revenue total anual de Alpha para 2024 (47.3 millones de euros), pero no incluyen un desglose trimestral que permita conocer específicamente los ingresos del tercer trimestre.'

In [15]:
# Evalúa cuántas veces el chunk recuperado realmente contiene la respuesta
test_cases = [
    {
        "pregunta": "¿Cuál es el margen EBITDA de Alpha?",
        "respuesta_esperada": "25.6",
        "fuente_esperada": "empresa_alpha_2024.txt"
    },
    {
        "pregunta": "¿Cuántos clientes activos tiene Gamma?",
        "respuesta_esperada": "342",
        "fuente_esperada": "empresa_gamma_2024.txt"
    },
    {
        "pregunta": "¿Qué porcentaje del revenue de Beta viene de manufactura?",
        "respuesta_esperada": "63",
        "fuente_esperada": "empresa_beta_2024.txt"
    },
    {
        "pregunta": "¿Cuál es el churn rate de Gamma?",
        "respuesta_esperada": "4.2",
        "fuente_esperada": "empresa_gamma_2024.txt"
    },
]

correctos = 0
for tc in test_cases:
    results = collection.query(query_texts=[tc["pregunta"]], n_results=1)
    fuente_recuperada = results["metadatas"][0][0]["source"]
    chunk_recuperado = results["documents"][0][0]
    
    fuente_ok = tc["fuente_esperada"] in fuente_recuperada
    dato_ok = tc["respuesta_esperada"] in chunk_recuperado
    
    status = "OK" if (fuente_ok and dato_ok) else "FALLO"
    if status == "OK":
        correctos += 1
    
    print(f"[{status}] {tc['pregunta'][:50]}...")
    if status == "FALLO":
        print(f"       Fuente esperada: {tc['fuente_esperada']}")
        print(f"       Fuente recuperada: {fuente_recuperada}")
        print(f"       Dato '{tc['respuesta_esperada']}' en chunk: {dato_ok}")

print(f"\nPrecision@1: {correctos}/{len(test_cases)} = {correctos/len(test_cases):.0%}")

[FALLO] ¿Cuál es el margen EBITDA de Alpha?...
       Fuente esperada: empresa_alpha_2024.txt
       Fuente recuperada: empresa_alpha_2024.txt
       Dato '25.6' en chunk: False
[OK] ¿Cuántos clientes activos tiene Gamma?...
[OK] ¿Qué porcentaje del revenue de Beta viene de manuf...
[OK] ¿Cuál es el churn rate de Gamma?...

Precision@1: 3/4 = 75%


In [ ]:
# Estos últimos resultados se deben a que he escogido 10 palabras por chunk, lo cual es demasiado pequeño, pero quería testear.